# 04 — Visualisation

Build the charts and network graphs shown in the Streamlit dashboard.

**What this notebook covers**
- Entity frequency charts
- Co-occurrence network (NetworkX + pyvis)
- ICD-10 heatmap
- Severity breakdown by specialty

## 1. Setup

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.nlp.ner import build_ner_pipeline
from src.nlp.cooccurrence import build_cooccurrence_graph, graph_summary
from src.utils.text_utils import clean_clinical_text
from src.etl.extract import load_mtsamples
from src.etl.transform import prepare_for_training

print('Setup OK')

Setup OK


## 2. Build Entity Dataset

Run NER on 300 notes to build a representative entity corpus.

In [2]:
ner = build_ner_pipeline()
df  = prepare_for_training(load_mtsamples())

sample = df.head(300)
note_entities = {}

for idx, row in sample.iterrows():
    ents = ner.extract(row['transcription'])
    if ents:
        note_entities[idx] = ents

total = sum(len(v) for v in note_entities.values())
print(f'Notes processed : {len(note_entities)}')
print(f'Total entities  : {total}')

2026-06-25 23:00:40 | INFO     | src.etl.extract                | Loading MTSamples from C:\Users\Hp\Documents\clinical-nlp-pipeline\data\raw\mtsamples.csv
2026-06-25 23:00:41 | INFO     | src.etl.extract                | MTSamples loaded: 4999 notes, 40 specialties
2026-06-25 23:00:41 | INFO     | src.etl.transform              | Running full transformation pipeline...
2026-06-25 23:00:41 | INFO     | src.etl.transform              | Dropped 33 rows with missing transcription text
2026-06-25 23:00:59 | INFO     | src.etl.transform              | Removed 2609 duplicate notes
2026-06-25 23:00:59 | INFO     | src.etl.transform              | Clean notes: 2357 rows remaining
2026-06-25 23:00:59 | INFO     | src.etl.transform              | Filtered 33 notes shorter than 30 words
2026-06-25 23:00:59 | INFO     | src.etl.transform              | Deriving severity labels via weak supervision...
2026-06-25 23:01:05 | INFO     | src.etl.transform              |   urgent     1053  (45.3%)
2026-

## 3. Entity Frequency Chart

In [3]:
from collections import Counter

all_ents  = [e for ents in note_entities.values() for e in ents]
by_label  = {}
for label in ['DISEASE', 'MEDICATION', 'PROCEDURE', 'SYMPTOM']:
    texts     = [e.text.lower() for e in all_ents if e.label == label]
    top15     = Counter(texts).most_common(15)
    by_label[label] = top15

label = 'DISEASE'
top   = by_label[label]
fig = px.bar(
    x=[c for _, c in top],
    y=[t for t, _ in top],
    orientation='h',
    title=f'Top 15 {label} entities',
    labels={'x': 'Frequency', 'y': ''},
    height=420,
)
fig.update_layout(yaxis={'autorange': 'reversed'})
fig.show()

## 4. Co-occurrence Network

In [4]:
graph = build_cooccurrence_graph(
    note_entities = note_entities,
    entity_label  = 'DISEASE',
    min_count     = 3,
    max_nodes     = 40,
)

summary = graph_summary(graph)
print(f'Nodes   : {summary["nodes"]}')
print(f'Edges   : {summary["edges"]}')
print(f'Density : {summary["density"]}')
print(f'\nTop entities by connections:')
for name, deg in summary['top_entities'][:8]:
    print(f'  {name:<30} {deg} connections')

2026-06-25 23:03:25 | INFO     | src.nlp.cooccurrence           | Co-occurrence graph: 500 nodes, 30 edges (min_count=3)
2026-06-25 23:03:25 | INFO     | src.nlp.cooccurrence           | Graph pruned to 40 nodes (max_nodes=40)
Nodes   : 40
Edges   : 30
Density : 0.0385

Top entities by connections:
  infection                      10 connections
  hypertension                   8 connections
  hernia                         4 connections
  tumor                          4 connections
  asthma                         4 connections
  dvt                            3 connections
  myocardial infarction          3 connections
  gastroesophageal reflux disease 2 connections


In [5]:
# Interactive pyvis network (saved as HTML, open in browser)
from src.nlp.cooccurrence import graph_to_pyvis

html = graph_to_pyvis(graph)
if html:
    out_path = '../data/processed/cooccurrence_graph.html'
    with open(out_path, 'w') as fh:
        fh.write(html)
    print(f'Interactive graph saved to {out_path}')
else:
    print('pyvis not installed — run: pip install pyvis')

Interactive graph saved to ../data/processed/cooccurrence_graph.html


## 5. Severity by Specialty

In [8]:
# Exclude document-type / administrative categories that are not genuine
# clinical specialties (MTSamples mixes these into the same
# specialty field, e.g. Soap / Chart / Progress Notes can come from
# any clinical domain, so comparing severity across them mixes
# note-type with clinical-domain rather than comparing specialties).
_NON_SPECIALTY_CATEGORIES = {
    'Soap / Chart / Progress Notes',
    'Consult - History And Phy.',
    'Office Notes',
    'Discharge Summary',
    'Letters',
    'Ime-Qme-Work Comp Etc.',
}
specialty_df = df[~df['specialty_clean'].isin(_NON_SPECIALTY_CATEGORIES)]

# Filter to the actual top 12 by note count -- categoryarray alone only
# controls display ORDER, it does not exclude unlisted categories, so
# without this filter every remaining specialty still gets plotted.
top12 = specialty_df['specialty_clean'].value_counts().head(12).index.tolist()
specialty_df_top12 = specialty_df[specialty_df['specialty_clean'].isin(top12)]

fig = px.histogram(
    specialty_df_top12,
    x         = 'specialty_clean',
    color     = 'severity',
    barmode   = 'group',
    title     = 'Severity distribution by specialty (top 12, clinical specialties only)',
    category_orders = {
        'specialty_clean': top12,
        'severity': ['routine', 'urgent', 'critical'],
    },
    color_discrete_map = {
        'routine':  '#2ecc71',
        'urgent':   '#f39c12',
        'critical': '#e74c3c',
    },
    height = 420,
)
fig.update_layout(xaxis = dict(tickangle=-30))
fig.show()

## 6. Known Limitations

This pipeline was hardened through extensive iterative bug-fixing. The remaining
known gaps are documented here rather than hidden, so the system's
actual failure modes are visible to anyone evaluating it.

### NER / entity classification

- **SYMPTOM precision is structurally weak (~12% on a hand-reviewed
  sample, see notebook 01 section 6).** SYMPTOM is the default bucket
  for any `en_core_sci_lg` entity span that doesn't match a more
  specific rule. It absorbs generic verbs ("evaluate", "decision"),
  connector phrases ("consistent with", "secondary to"), and equipment
  names ("stockinette", "guidewires") that the underlying model
  extracts but that aren't clinically meaningful entities. Fixing this
  properly would need either a trained classifier or POS-tag-based
  filtering, not more entries in a keyword list — the keyword-list
  approach has diminishing returns against this kind of open-ended
  noise.
- **NER span-boundary errors are not fixable via post-processing.**
  The underlying spaCy/scispaCy models occasionally produce spans that
  truncate ("trace pulmonary" missing "insufficiency") or merge across
  a sentence/section boundary ("room cancer" from "emergency room" +
  "cancer"). A punctuation-run heuristic catches the worst merge cases,
  but this is a model-level limitation, not a classification rule gap.
- **Ambiguous abbreviations resolve to one meaning only.** "MCA" is
  filtered entirely because it's anatomy (middle cerebral artery) in
  this corpus but could mean something else elsewhere. "PCP" is treated
  as "primary care physician" and filtered; it can also mean
  phencyclidine. There's no context-aware disambiguation.
- **ICD-10 chapter classification (R-code = symptom) has known
  exceptions.** Joint/limb pain is coded under musculoskeletal M-codes,
  not R-codes — handled via an explicit anatomy+qualifier override, but
  any other exception category not yet encountered would still be
  misclassified the same way until found.
- **bc5cdr's CHEMICAL tag is over-trusted by default.** It also tags
  lab-test names ("cholesterol", "glucose", "ANA") and social-history
  substances ("alcohol", "tobacco") as CHEMICAL. A skip-list catches
  the specific instances found during review; it is not an exhaustive
  fix for the category.
- **No UMLS/SNOMED-based concept normalisation.** Synonym/abbreviation
  pairs are merged only where specifically discovered and added to the
  abbreviation-expansion dictionary (e.g. GERD). A proper concept-linking
  layer would catch this class of duplication systematically instead of
  one pair at a time — deliberately out of scope for this project given
  the ~1GB+ dependency and added complexity it would require.

### Classification (severity model)

- **Critical-class recall is weak (48.6% recall, F1 0.515 vs 0.75+ for
  the other two classes)** — see notebook 03 section 3b. Deferred by
  design decision: class-weighted loss or oversampling are the standard
  fixes, not yet applied given retraining cost (60–90+ min on this
  CPU-only machine).
- **Severity labels are weak supervision, not ground truth.** They come
  from keyword-rule heuristics (`src/etl/transform.py`), not clinician
  annotation. The classifier's reported accuracy is bounded by how good
  those heuristic labels are, not by true diagnostic accuracy.

### Evaluation methodology

- **The gold-standard precision numbers in notebook 01 are a
  self-review**, not an independent clinical annotation. The same
  person who wrote the classification rules also judged whether they
  were correct, which is a meaningfully weaker standard of evidence
  than third-party-annotated benchmarks. Treat it as a sanity check,
  not a publishable metric.
- **Single data source.** Everything here is trained and evaluated on
  MTSamples only — a fixed, US-centric, de-identified transcription
  corpus. Generalisation to other note formats, institutions, or
  populations is untested.
